# GCascadeV5 Tutorial Notebook

Welcome to the GCascadeV5 tutorial. This notebook is intended to be beginner-friendly: it explains *what each function does*, *what units it expects*, and *how to format arrays correctly*.

This notebook covers setup, the main propagation modes (point, diffuse, evolving), the gamma/electron cascade runtime, optional electron injection, diagnostic electron outputs, and the main controls for EBL model and magnetic field.


## 1) Required packages

Required runtime packages:
- `numpy`
- `scipy`
- `h5py`
- `matplotlib`

Required to run this tutorial notebook:
- `notebook` (Jupyter Notebook)

`pip install -e '.[dev]'` installs GCascadeV5 in editable mode, the runtime packages, the test tools, and Jupyter Notebook.

Installation commands:
```bash
cd /path/to/GCascadeV5
python3 -m venv .venv
source .venv/bin/activate
pip install -e '.[dev]'
jupyter notebook tutorial.ipynb
```


## 2) Prerequisites and execution notes

- Run notebook cells in order.
- Inputs and outputs use **physical units** (detailed below).
- All source redshifts must satisfy `0 <= z <= 10`.
- Before running attenuation/cascade functions, confirm that GCascadeV5 points to your local `LibrariesV5` directory.
- The main `Cascade*` APIs now use schema-v2 HDF5 tables with integrated PP and ICS kernels stored directly in `LibrariesV5`.
- Regular runtime calls read everything they need from `LibrariesV5`; there is no separate external table directory in the default workflow.


## 3) Configure library paths (first step)

GCascadeV5 uses one main configurable path:

- `library_path`: the `LibrariesV5` directory containing the runtime HDF5 tables.

The standard `LibrariesV5` bundle already contains the pair-production and inverse-Compton tables needed for the gamma/electron cascade.

Set the path inside GCascadeV5 after import:

- `gc.set_library_path('/absolute/path/to/LibrariesV5')`

The next cell prints the active path and progress-bar status so you can verify the configuration immediately.

Optional environment control:
- `GCASCADE_PROGRESS=0` disables progress bars.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import gcascade_v5 as gc

# Configure the runtime bundle after import (edit and uncomment as needed)
# gc.set_library_path('/absolute/path/to/LibrariesV5')

print('GCascadeV5 version:', gc.__version__)
print('Active library path:', gc.get_library_path())
print('Progress bars enabled:', gc.PROGRESS_ENABLED)

if not Path(gc.get_library_path()).exists():
    print('WARNING: library path does not exist yet. Set it with gc.set_library_path(...)')


## 4) Core arrays and what they mean

GCascadeV5 exposes key grids used by all computations:

- `energies`: gamma-ray energy grid in **GeV**.
  - Length: 300
  - Log-spaced from `1e-1 GeV` to `1e12 GeV`
- `diffuseDistances`: redshift grid used for source-population integrals and propagation slicing.
  - Length: 1036
- `zReg`: coarser redshift regions (`0` to `10` in steps of `0.01`) used to index precomputed interaction tables.

In [ ]:
print('len(energies) =', len(gc.energies))
print('energies[0], energies[-1] =', gc.energies[0], gc.energies[-1], 'GeV')
print('len(diffuseDistances) =', len(gc.diffuseDistances))
print('diffuseDistances min/max =', gc.diffuseDistances[0], gc.diffuseDistances[-1])
print('len(zReg) =', len(gc.zReg))
print('current EBL index =', gc.EBLindex)

## 5) Physical unit conventions

### Injected gamma and electron spectra
For point and diffuse non-evolving functions, injected spectra should be:

- `dN/dE(E)` in **GeV^-1 s^-1**
- Evaluated on `gc.energies`
- Shape `(300,)`

Gamma injection is the first positional argument. Electron injection is optional:

- `CascadePoint(gamma_inj, z_source, electronSpectraPre=electron_inj)`
- `CascadeDiffuse(gamma_inj, z_max, zDistrib, electronSpectra=electron_inj)`

For evolving functions, use one spectrum per redshift grid point:

- `gamma_inj2d[z_index, e_index] = dN/dE(E, z)`
- optional `electronSpectra=electron_inj2d`
- Shape `(len(diffuseDistances), len(energies)) = (1036, 300)`

### Source-density distribution
For diffuse/evolving calculations, `zDistrib` should be:

- `rho(z)` in **cm^-3**
- Evaluated on `gc.diffuseDistances`
- Shape `(1036,)`

### Typical output units
- Point-source gamma outputs: **GeV^-1 s^-1 cm^-2**
- Diffuse / evolving gamma outputs: **GeV^-1 s^-1 cm^-2 sr^-1**

By default, cascade functions still return only the final gamma-ray spectrum. Pass `return_state=True` to get a `CascadeResult` with:

- `result.gamma`: final gamma-ray spectrum
- `result.electron`: final extragalactic electron boundary-state spectrum
- `result.diagnostics`: energy-accounting diagnostics
- `result.metadata`: run metadata

The electron output is mainly a diagnostic/internal transport product. It is not a directly observable charged-particle flux at Earth because magnetic deflections, local Galactic propagation, and angular/time information are outside the scope of this runtime.


## 6) Build valid injected spectra

`cutoffPowerLaw(E, gamma, cutoff, amp)` is a convenience helper:

`dN/dE = amp * E^(-gamma) * exp(-E/cutoff)`

where `E` and `cutoff` are in GeV.

Here we build a gamma-ray injection spectrum and, separately, an optional electron injection spectrum on the same energy grid.


In [ ]:
gamma_inj = gc.cutoffPowerLaw(gc.energies, gamma=2.2, cutoff=1e7, amp=1e40)
electron_inj = gc.cutoffPowerLaw(gc.energies, gamma=2.4, cutoff=1e6, amp=1e41)

# Backward-compatible name used by later cells.
inj = gamma_inj

print('gamma shape:', gamma_inj.shape)
print('electron shape:', electron_inj.shape)
print('gamma min/max:', gamma_inj.min(), gamma_inj.max())
print('electron min/max:', electron_inj.min(), electron_inj.max())


In [ ]:
def set_relevant_plot_window(ax, *spectra, span_decades=10):
    """Keep the visible y-range focused on the physically relevant part of each spectrum."""
    x = np.asarray(gc.energies, dtype=float)
    curves = []
    for spec in spectra:
        y = x**2 * np.asarray(spec, dtype=float)
        mask = np.isfinite(y) & (y > 0.0)
        if np.any(mask):
            curves.append(y)
    if not curves:
        return

    peak = max(float(np.max(y[np.isfinite(y) & (y > 0.0)])) for y in curves)
    floor = peak / (10.0**span_decades)
    significant = np.zeros_like(x, dtype=bool)
    kept_values = []
    for y in curves:
        mask = np.isfinite(y) & (y >= floor)
        significant |= mask
        if np.any(mask):
            kept_values.append(y[mask])
    if not kept_values:
        return

    ymin = max(float(np.min(np.concatenate(kept_values))) / 1.5, floor)
    ax.set_ylim(ymin, peak * 1.5)
    if np.any(significant):
        last = np.where(significant)[0][-1]
        ax.set_xlim(x[0], x[min(last + 2, len(x) - 1)])

fig, ax = gc.specPlot(gamma_inj)
ax.loglog(gc.energies, gc.energies**2 * electron_inj, ls='--', label='optional electron injection')
ax.set_title('Injected spectra: E^2 dN/dE')
set_relevant_plot_window(ax, gamma_inj, electron_inj)
ax.legend()


## 7) Point-source propagation

For a source at redshift `z_source`, GCascade provides three levels:

- `RedshiftPoint(gamma_inj, z_source)`: only cosmological redshifting
- `AttenuatePoint(gamma_inj, z_source)`: redshifting + pair-production attenuation
- `CascadePoint(gamma_inj, z_source, electronSpectraPre=None, return_state=False)`: redshifting + attenuation + gamma/electron cascade transport

`CascadePoint` remains backward-compatible: without `return_state=True`, it returns only the final gamma-ray spectrum. With `return_state=True`, it returns a `CascadeResult` containing gamma, electron, diagnostics, and metadata. For sources at nonzero redshift, the diagnostics also report `redshift_energy_lost`, which is the physically expected energy drain from cosmological expansion. They also include `energy_residual` and `relative_energy_residual`, where the latter is normalized to the total injected gamma + electron energy.


In [ ]:
z_source = 0.3

phi_redshift = gc.RedshiftPoint(gamma_inj, z_source)
phi_atten = gc.AttenuatePoint(gamma_inj, z_source)
phi_cascade = gc.CascadePoint(gamma_inj, z_source)

point_state = gc.CascadePoint(
    gamma_inj,
    z_source,
    electronSpectraPre=electron_inj,
    return_state=True,
)

print(phi_redshift.shape, phi_atten.shape, phi_cascade.shape)
print(type(point_state).__name__)
print(point_state.gamma.shape, point_state.electron.shape)
print(point_state.diagnostics)
print('relative energy residual:', point_state.diagnostics['relative_energy_residual'])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(gc.energies, gc.energies**2 * phi_redshift, label='RedshiftPoint')
ax.loglog(gc.energies, gc.energies**2 * phi_atten, label='AttenuatePoint')
ax.loglog(gc.energies, gc.energies**2 * phi_cascade, label='CascadePoint (gamma-only injection)')
ax.loglog(gc.energies, gc.energies**2 * point_state.gamma, ls='--', label='CascadePoint (gamma + electron injection)')
ax.set_xlabel('E [GeV]')
ax.set_ylabel(r'$E^2 \, \phi(E)$')
ax.set_title('Point-source transport comparison')
ax.legend()
ax.grid(True, which='both', alpha=0.25)
set_relevant_plot_window(ax, phi_redshift, phi_atten, phi_cascade, point_state.gamma)


## 8) Diffuse non-evolving source population

Diffuse functions assume all sources share one intrinsic gamma spectrum (`gamma_inj`) and are distributed in redshift via `zDistrib`.

Required shapes:
- `gamma_inj`: `(300,)`
- optional `electronSpectra`: `(300,)`
- `zDistrib`: `(1036,)`

Main APIs:
- `RedshiftDiffuse(gamma_inj, z_max, zDistrib)`
- `AttenuateDiffuse(gamma_inj, z_max, zDistrib)`
- `CascadeDiffuse(gamma_inj, z_max, zDistrib, electronSpectra=None, return_state=False)`

The electron spectrum, when supplied, is injected by the same diffuse source population and weighted by the same `zDistrib` normalization.


In [ ]:
z = gc.diffuseDistances
z_distrib = 5e-6 * (3.086e24)**-3 * (1.0 + z)**-2  # cm^-3

# Keep tutorial execution light. Increase z_max for production runs.
z_max_demo = 0.1

phi_diff_r = gc.RedshiftDiffuse(gamma_inj, z_max_demo, z_distrib)
phi_diff_a = gc.AttenuateDiffuse(gamma_inj, z_max_demo, z_distrib)
phi_diff_c = gc.CascadeDiffuse(gamma_inj, z_max_demo, z_distrib)
phi_diff_state = gc.CascadeDiffuse(
    gamma_inj,
    z_max_demo,
    z_distrib,
    electronSpectra=electron_inj,
    return_state=True,
)

print(
    phi_diff_r.shape,
    phi_diff_a.shape,
    phi_diff_c.shape,
    phi_diff_state.gamma.shape,
    phi_diff_state.electron.shape,
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(gc.energies, gc.energies**2 * phi_diff_r, label='RedshiftDiffuse')
ax.loglog(gc.energies, gc.energies**2 * phi_diff_a, label='AttenuateDiffuse')
ax.loglog(gc.energies, gc.energies**2 * phi_diff_c, label='CascadeDiffuse (gamma-only injection)')
ax.loglog(gc.energies, gc.energies**2 * phi_diff_state.gamma, ls='--', label='CascadeDiffuse (gamma + electron injection)')
ax.set_xlabel('E [GeV]')
ax.set_ylabel(r'$E^2 \, \phi(E)$')
ax.set_title('Diffuse non-evolving transport comparison')
ax.legend()
ax.grid(True, which='both', alpha=0.25)
set_relevant_plot_window(ax, phi_diff_r, phi_diff_a, phi_diff_c, phi_diff_state.gamma)


## 9) Diffuse evolving source population

For evolving populations, gamma injection is a 2D array:

- `gamma_inj2d[z_index, e_index] = dN/dE(E, z)`
- Shape `(len(diffuseDistances), len(energies)) = (1036, 300)`

Optional electron injection has the same 2D shape:

- `electron_inj2d[z_index, e_index] = dN_e/dE(E, z)`

Main APIs:
- `RedshiftEvolving(gamma_inj2d, z_max, zDistrib)`
- `AttenuateEvolving(gamma_inj2d, z_max, zDistrib)`
- `CascadeEvolving(gamma_inj2d, z_max, zDistrib, electronSpectra=None, return_state=False)`


In [ ]:
gamma_inj2d = np.empty((len(gc.diffuseDistances), len(gc.energies)))
electron_inj2d = np.empty_like(gamma_inj2d)

for i, zi in enumerate(gc.diffuseDistances):
    gamma_i = 2.0 + 0.2 * zi / 10.0
    gamma_inj2d[i] = gc.cutoffPowerLaw(gc.energies, gamma=gamma_i, cutoff=1e7, amp=1e40)
    electron_inj2d[i] = gc.cutoffPowerLaw(gc.energies, gamma=gamma_i + 0.2, cutoff=1e6, amp=1e34)

z_max_demo = 0.1

phi_ev_r = gc.RedshiftEvolving(gamma_inj2d, z_max_demo, z_distrib)
phi_ev_a = gc.AttenuateEvolving(gamma_inj2d, z_max_demo, z_distrib)
phi_ev_c = gc.CascadeEvolving(gamma_inj2d, z_max_demo, z_distrib)
phi_ev_state = gc.CascadeEvolving(
    gamma_inj2d,
    z_max_demo,
    z_distrib,
    electronSpectra=electron_inj2d,
    return_state=True,
)

print(
    phi_ev_r.shape,
    phi_ev_a.shape,
    phi_ev_c.shape,
    phi_ev_state.gamma.shape,
    phi_ev_state.electron.shape,
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(gc.energies, gc.energies**2 * phi_ev_r, label='RedshiftEvolving')
ax.loglog(gc.energies, gc.energies**2 * phi_ev_a, label='AttenuateEvolving')
ax.loglog(gc.energies, gc.energies**2 * phi_ev_c, label='CascadeEvolving (gamma-only injection)')
ax.loglog(gc.energies, gc.energies**2 * phi_ev_state.gamma, ls='--', label='CascadeEvolving (gamma + electron injection)')
ax.set_xlabel('E [GeV]')
ax.set_ylabel(r'$E^2 \, \phi(E)$')
ax.set_title('Diffuse evolving transport comparison')
ax.legend()
ax.grid(True, which='both', alpha=0.25)
set_relevant_plot_window(ax, phi_ev_r, phi_ev_a, phi_ev_c, phi_ev_state.gamma)


## 10) Advanced controls

### 10.1 Change EBL model
EBL index map:
- `0`: CMB only
- `1`: Saldana-Lopez 2021 (default)
- `2`: Saldana-Lopez high
- `3`: Saldana-Lopez low
- `4`: Finke 2022
- `5`: Franceschini & Rodighiero 2018
- `6`: Dominguez 2011

All EBL runtime files in a schema-v2 bundle should include PP and ICS tables. Changing EBL model switches both the gamma attenuation data and the electron/ICS transport data.


In [ ]:
old_ebl = gc.EBLindex
if old_ebl != 4:
    gc.changeEBLModel(4)
print('active EBL:', gc.EBLindex)
print('bundle info:', gc.get_bundle_info())

# restore
if gc.EBLindex != old_ebl:
    gc.changeEBLModel(old_ebl)


### 10.2 Change magnetic field
`changeMagneticField(Bfield, gamma, EBLindex)` configures synchrotron-loss competition in the electron-tracking cascade.

- `Bfield` in **Gauss**
- field scaling: `B(z) = Bfield * (1+z)^gamma`
- after `changeMagneticField(..., EBLindex)`, that EBL model becomes the active one
- synchrotron energy removed from the electron cascade is reported in `result.diagnostics['synchrotron_energy_lost']`
- use `gc.reset_factory_settings()` to restore the default EBL model and the zero magnetic-field setting


In [ ]:
# Example: set an extragalactic magnetic field for electron transport.
# This changes synchrotron-loss accounting in subsequent Cascade* calls.
old_ebl = gc.EBLindex

gc.changeMagneticField(1e-18, 0.0, old_ebl)
state_with_b = gc.CascadePoint(gamma_inj, 0.1, electronSpectraPre=electron_inj, return_state=True)
print('synchrotron energy lost:', state_with_b.diagnostics['synchrotron_energy_lost'])

# Return to the default EBL/cycle/magnetic-field settings.
gc.reset_factory_settings()
if old_ebl != gc.EBLindex:
    gc.changeEBLModel(old_ebl)


## 11) Exporting results

A common workflow is to export `(energy, flux)` columns for external plotting/fitting.

For backward-compatible calls, export the returned gamma spectrum as before. If you used `return_state=True`, export `result.gamma` for the gamma-ray observable and optionally export `result.electron` as a diagnostic extragalactic boundary-state spectrum.


In [ ]:
point_gamma_table = np.column_stack([gc.energies, point_state.gamma])
point_electron_table = np.column_stack([gc.energies, point_state.electron])

np.savetxt(
    'point_cascade_gamma_flux.csv',
    point_gamma_table,
    delimiter=',',
    header='E_GeV,Phi_gamma_GeV^-1_s^-1_cm^-2',
    comments='',
)
np.savetxt(
    'point_cascade_electron_diagnostic.csv',
    point_electron_table,
    delimiter=',',
    header='E_GeV,Electron_boundary_state_GeV^-1_s^-1_cm^-2',
    comments='',
)

print('saved point_cascade_gamma_flux.csv')
print('saved point_cascade_electron_diagnostic.csv')
